# Kasus B: Sentimen Ulasan Produk (Video Games)

Klasifikasi biner ulasan Amazon Video Games: **negatif** (rating 1-2) vs **positif** (rating 4-5), rating 3 dibuang.
Metrik: **macro-F1** (rata-rata F1 kelas negatif dan positif).

Alur: baca data (streaming) -> split berdasarkan waktu -> TF-IDF + Logistic Regression -> tuning threshold -> simpan model.

In [1]:
import gzip, json, random, time
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline

## 1. Konfigurasi

- Ulasan dengan timestamp **setelah 31 Des 2019 (UTC)** dibuang. Timestamp di file berupa **milidetik**.
- `SAMPLE_RATE`: file aslinya sangat besar, jadi kita ambil sampel acak per baris (distribusi kelas tetap terjaga). Naikkan ke `1.0` kalau RAM cukup dan mau pakai semua data.
- Split berdasarkan waktu: **latih** = ulasan sebelum 2019, **validasi** = ulasan tahun 2019.

In [2]:
DATA_PATH   = "data/Video_Games.jsonl.gz"
MODEL_PATH  = "models/sentiment_model.joblib"
SAMPLE_RATE = 0.15      # 1.0 = pakai semua data
SEED        = 42

def to_ms(y, m, d):
    return int(datetime(y, m, d, tzinfo=timezone.utc).timestamp() * 1000)

CUTOFF_MS = to_ms(2020, 1, 1)   # timestamp harus < 1 Jan 2020 00:00 UTC (= sampai 31 Des 2019)
VAL_START = to_ms(2019, 1, 1)   # validasi = tahun 2019
print(CUTOFF_MS, VAL_START)

1577836800000 1546300800000


## 2. Membaca data per baris (streaming)

File dibaca satu baris demi satu langsung dari `.gz` (tanpa di-extract dan tanpa dimuat utuh).
Sampling dilakukan **sebelum** parsing JSON supaya cepat. Label: rating <= 2 -> 0 (negatif), rating >= 4 -> 1 (positif), rating 3 dibuang.

In [3]:
def load_reviews(path, sample_rate, seed=SEED):
    rng = random.Random(seed)
    rows, n_lines = [], 0
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            n_lines += 1
            if rng.random() > sample_rate:
                continue
            r = json.loads(line)
            ts, rating = r.get("timestamp"), r.get("rating")
            if ts is None or rating is None or ts >= CUTOFF_MS:
                continue
            if rating <= 2:
                y = 0
            elif rating >= 4:
                y = 1
            else:
                continue
            text = ((r.get("title") or "") + ". " + (r.get("text") or "")).strip()
            if len(text) < 3:
                continue
            rows.append((ts, text, y))
    print(f"baris dibaca: {n_lines:,} | dipakai: {len(rows):,}")
    return pd.DataFrame(rows, columns=["timestamp", "text", "label"])

t0 = time.time()
df = load_reviews(DATA_PATH, SAMPLE_RATE)
print(f"selesai dalam {time.time()-t0:.0f} detik")
df.head()

baris dibaca: 4,624,615 | dipakai: 422,211
selesai dalam 18 detik


,timestamp,text,label
0,1562177742305,"DONT BUY. NOT RECCOMENDED, BROKE 2MONTYS AFTER...",0
1,1394573240000,Great Game. This is one of the better Pokemon ...,1
2,1394573186000,Excellent Device. I am really enjoying this de...,1
3,1574659954094,Not too bad for the price.. This is an ok head...,1
4,1496373398000,interesting story. I picked this book up at a ...,1


In [4]:
print("rentang tanggal:",
      datetime.fromtimestamp(df.timestamp.min()/1000, timezone.utc).date(), "s.d.",
      datetime.fromtimestamp(df.timestamp.max()/1000, timezone.utc).date())
print(df.label.value_counts().rename({0: "negatif", 1: "positif"}))
print(df.label.value_counts(normalize=True).round(3).rename({0: "negatif", 1: "positif"}))

rentang tanggal: 1999-11-10 s.d. 2019-12-31
label
positif    348405
negatif     73806
Name: count, dtype: int64
label
positif    0.825
negatif    0.175
Name: proportion, dtype: float64


## 3. Split latih / validasi (berdasarkan waktu)

Model nanti dipakai untuk ulasan yang **baru masuk**, jadi validasinya sebaiknya meniru itu: latih di ulasan yang lebih lama, uji di ulasan yang lebih baru.
Split acak akan "membocorkan" tren dan gaya bahasa periode yang sama ke data latih sehingga skornya terlalu optimis.

In [5]:
train = df[df.timestamp <  VAL_START].reset_index(drop=True)
val   = df[df.timestamp >= VAL_START].reset_index(drop=True)
print(f"latih: {len(train):,} | validasi: {len(val):,} ({len(val)/len(df):.1%})")
print("proporsi negatif -> latih:", round((train.label==0).mean(),3), "| validasi:", round((val.label==0).mean(),3))

latih: 359,082 | validasi: 63,129 (15.0%)
proporsi negatif -> latih: 0.175 | validasi: 0.174


## 4. Model: TF-IDF + Logistic Regression

- TF-IDF unigram + bigram (menangkap frasa seperti "not good", "waste of money").
- `class_weight="balanced"` supaya kelas negatif (lebih sedikit) tidak diabaikan.

In [6]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=300_000,
                              sublinear_tf=True, strip_accents="unicode")),
    ("clf",   LogisticRegression(C=4.0, class_weight="balanced", solver="liblinear")),
])

t0 = time.time()
pipe.fit(train.text, train.label)
print(f"training selesai dalam {time.time()-t0:.0f} detik")

training selesai dalam 81 detik


## 5. Evaluasi di data validasi

Ditampilkan macro-F1 dengan threshold default (0.5), lalu dicari threshold yang memaksimalkan macro-F1.
Catatan: threshold dipilih di data validasi yang sama dengan tempat skor dilaporkan, jadi skornya bisa sedikit optimis.

In [7]:
val_proba = pipe.predict_proba(val.text)[:, 1]   # peluang kelas positif

def macro_f1(y, proba, thr):
    return f1_score(y, (proba >= thr).astype(int), average="macro")

f1_default = macro_f1(val.label, val_proba, 0.5)
thresholds = np.arange(0.20, 0.80, 0.01)
scores = [macro_f1(val.label, val_proba, t) for t in thresholds]
best_thr = float(thresholds[int(np.argmax(scores))])
f1_tuned = max(scores)

print(f"macro-F1 (threshold 0.50): {f1_default:.4f}")
print(f"macro-F1 (threshold {best_thr:.2f}): {f1_tuned:.4f}")

macro-F1 (threshold 0.50): 0.9402
macro-F1 (threshold 0.33): 0.9460


In [8]:
pred = (val_proba >= best_thr).astype(int)
print(classification_report(val.label, pred, target_names=["negatif", "positif"], digits=4))

              precision    recall  f1-score   support

     negatif     0.9062    0.9156    0.9109     11010
     positif     0.9821    0.9800    0.9811     52119

    accuracy                         0.9688     63129
   macro avg     0.9442    0.9478    0.9460     63129
weighted avg     0.9689    0.9688    0.9688     63129



## 6. Analisis hasil

Tiga hal yang dicek: (a) confusion matrix, (b) kata yang paling memengaruhi prediksi, (c) contoh ulasan yang salah ditebak.

In [9]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(val.label, pred)
print(pd.DataFrame(cm, index=["asli negatif", "asli positif"], columns=["tebak negatif", "tebak positif"]))

              tebak negatif  tebak positif
asli negatif          10081            929
asli positif           1043          51076


In [10]:
vocab = np.array(pipe.named_steps["tfidf"].get_feature_names_out())
coef  = pipe.named_steps["clf"].coef_[0]
order = np.argsort(coef)
print("Kata paling menandakan NEGATIF:\n ", ", ".join(vocab[order[:25]]))
print()
print("Kata paling menandakan POSITIF:\n ", ", ".join(vocab[order[-25:][::-1]]))

Kata paling menandakan NEGATIF:
  two stars, one star, not, not worth, worst, terrible, disappointed, horrible, boring, disappointing, garbage, poor, disappointment, useless, at best, not good, not buy, awful, worse, waste, don buy, poorly, doesn, not very, broke

Kata paling menandakan POSITIF:
  great, awesome, best, perfect, four stars, amazing, excellent, five stars, love, not bad, fun, five, good, fantastic, perfectly, not disappointed, love it, no issues, nice, stars, no problems, my only, addictive, works, better than


In [11]:
err = val.assign(pred=pred, proba=val_proba)
err = err[err.label != err.pred].copy()
err["text"] = err.text.str.slice(0, 220)

for lbl, nama in [(0, "ASLI NEGATIF tapi ditebak POSITIF"), (1, "ASLI POSITIF tapi ditebak NEGATIF")]:
    sub = err[err.label == lbl]
    print(f"{nama}: {len(sub):,} kasus. Contoh acak:")
    for _, r in sub.sample(min(5, len(sub)), random_state=SEED).iterrows():
        print(f"  [p(positif)={r.proba:.2f}] {r.text}")
    print()

ASLI NEGATIF tapi ditebak POSITIF: 929 kasus. Contoh acak:
  [p(positif)=0.80] great screen protector but.... one of the  glass corner was cracked.
  [p(positif)=0.68] It's okay. I bought the switch for the fortnite skin, the console had already dead pixles.
  [p(positif)=0.69] There really isn't any of the good old rock/metal songs, it's pretty much all new stuff. The box it came in was fine, the shipping was fast, the games selection of music was terrible.
  [p(positif)=0.85] Know what you're getting yourself into. Wasn't easy to import files from other players to try their setup to customize your own setup be prepared to sit there an hour or two maybe four to get it right for that one car
  [p(positif)=1.00] Good price.. This will be my grandson's Xmas gift, so really can't say much until I get some feedback from him.

ASLI POSITIF tapi ditebak NEGATIF: 1,043 kasus. Contoh acak:
  [p(positif)=0.33] interesante. la posibilidades para configurarlo son demasiadas, el control tiene much

## 7. Simpan model

Yang disimpan: pipeline (TF-IDF + model) dan threshold terbaik, dalam satu file.

In [12]:
import os
os.makedirs("models", exist_ok=True)
joblib.dump({"pipeline": pipe, "threshold": best_thr}, MODEL_PATH)
print("tersimpan:", MODEL_PATH, f"({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)")

tersimpan: models/sentiment_model.joblib (13.3 MB)


## 8. Muat ulang model dan uji

Memastikan file model bisa dimuat ulang dan menghasilkan prediksi yang sama.

In [13]:
bundle = joblib.load(MODEL_PATH)
loaded_pipe, loaded_thr = bundle["pipeline"], bundle["threshold"]

def predict_sentiment(texts):
    proba = loaded_pipe.predict_proba(texts)[:, 1]
    return ["positif" if p >= loaded_thr else "negatif" for p in proba]

# hasil harus identik dengan model di memori
assert np.allclose(loaded_pipe.predict_proba(val.text[:1000]), pipe.predict_proba(val.text[:1000]))

contoh = [
    "Amazing game. Best purchase this year, totally worth it!",
    "Terrible. Constant crashes and a complete waste of money.",
    "Disc arrived scratched and the game would not even start.",
]
for t, s in zip(contoh, predict_sentiment(contoh)):
    print(f"{s:8s} <- {t}")

positif  <- Amazing game. Best purchase this year, totally worth it!
negatif  <- Terrible. Constant crashes and a complete waste of money.
negatif  <- Disc arrived scratched and the game would not even start.


## Kesimpulan

- Macro-F1 di data validasi (ulasan 2019): **0,9460** (threshold 0,33; pada threshold 0,50 skornya 0,9402).
- Model lebih kuat di kelas positif (F1 0,9811) dibanding negatif (F1 0,9109) karena kelas negatif lebih sedikit dan teksnya lebih ambigu.
- Kesalahan umumnya pada ulasan campuran, ulasan non-Inggris, dan label yang tidak sejalan dengan teks.
- Model memanfaatkan frasa seperti "one star" dan "five stars" yang ditulis pengulas sendiri.
- Ide perbaikan: pakai lebih banyak data (naikkan SAMPLE_RATE), model transformer, atau kumpulan data khusus Video Games tanpa aksesori.